# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maryam12arshad17/flyrank-internship-ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## 1: Two Paper Findings + Methodology Questions

I picked two ML-appendix findings from the paper, since these are the ones most
open to methodology questions (the paper itself notes "ML pages are exploratory
appendix material and do not override direct portfolio evidence").

---

**Finding A — "Which Dead Pages Can Come Back?" (Zombie Recovery model)**

The paper reports a Zombie Recovery model with 99% accuracy on same-brand
unseen pages and 97% on entirely unseen brands — the highest accuracy of any
model in the report. Top predictors listed are Content Age, Impressions, Days
Visible, Days Since Update.

*Methodology question:* Where exactly does the "recovery" label's outcome
window sit relative to the "Impressions" feature's measurement window? If
"Impressions" is measured over a period that overlaps with or comes after the
window used to decide whether a page "recovered," the feature could be
partially describing the same event as the label rather than predicting it —
which would explain why accuracy here (97-99%) is so much higher than the
other two models (75-90%). The report doesn't specify the exact date ranges
for feature vs. label windows for this particular model, so this is a
question I'd want answered before treating 97%+ as clean predictive skill
rather than partial window overlap.

---

**Finding B — "Which Pages Will Grow?" (Growth Prediction model)**

The paper reports 90% accuracy on same-brand pages but only 75% on unseen
brands (range 64%-85% across 20 tests) — a meaningfully wider same-brand vs.
new-brand gap than the other two models.

*Methodology question:* The paper doesn't say whether the underlying
"growing vs declining" label (used elsewhere in Finding 1 too) is defined by
comparing two time windows of the same metric the features also summarize
(e.g., impressions trend vs. "Impressions per Visible Day" as a feature). If
so, part of the same-brand 90% may reflect brand-specific patterns the model
memorized (client writing style, niche, publishing cadence) rather than a
generalizable growth signal — which the 15-point drop on unseen brands
partially confirms already. A useful addition would be reporting the label's
exact date range next to the feature windows, the way this notebook's
Section 2 tries to do for my own model.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

##  2: My model under an honest split (before/after)

My Week-5 model already used a client-grouped split, which is the honest choice
here — multiple content items repeat under the same client, so a random split
would let the model partly memorize client-specific patterns. Below I show
"before" (a naive random split, where rows from the same client can appear in
both train and test) vs "after" (the grouped split from ML-08), to make that
gap visible directly — the same kind of same-brand vs. new-brand comparison
the paper reports for its own models.

In [2]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"

print("Connected! Ready to query.")

Connected! Ready to query.


In [5]:
import pandas as pd

base = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions_total,
        SUM(gsc_clicks) AS gsc_clicks_total,
        AVG(gsc_avg_position) AS gsc_avg_position,
        SUM(ga4_sessions) AS ga4_sessions_total,
        SUM(ga4_pageviews) AS ga4_pageviews_total,
        SUM(ga4_engaged_sessions) AS ga4_engaged_sessions_total,
        COUNT(*) AS days_observed,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS days_gsc_available
    FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')
    GROUP BY client_hash_id, content_hash_id
""").df()

base["ctr"] = (base["gsc_clicks_total"] / base["gsc_impressions_total"]).fillna(0)
base["engagement_rate"] = (base["ga4_engaged_sessions_total"] / base["ga4_sessions_total"]).fillna(0)
base["gsc_coverage"] = (base["days_gsc_available"] / base["days_observed"]).fillna(0)

numeric_cols = ["gsc_impressions_total", "gsc_clicks_total", "gsc_avg_position",
                 "ga4_sessions_total", "ga4_pageviews_total", "ga4_engaged_sessions_total"]
base[numeric_cols] = base[numeric_cols].fillna(0)

label_data = con.sql(f"""
    WITH halves AS (
        SELECT client_hash_id, content_hash_id,
            SUM(CASE WHEN report_date < DATE '2026-03-16' THEN gsc_impressions ELSE 0 END) AS first_half,
            SUM(CASE WHEN report_date >= DATE '2026-03-16' THEN gsc_impressions ELSE 0 END) AS second_half
        FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT client_hash_id, content_hash_id, (second_half < first_half) AS is_declining
    FROM halves
""").df()

df = base.merge(label_data, on=["client_hash_id", "content_hash_id"])
print("Dataset shape:", df.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Dataset shape: (331437, 14)


In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

feature_cols = ["gsc_impressions_total", "gsc_clicks_total", "gsc_avg_position",
                 "ga4_sessions_total", "ga4_pageviews_total", "ga4_engaged_sessions_total",
                 "days_observed", "days_gsc_available", "ctr", "engagement_rate", "gsc_coverage"]

# --- BEFORE: naive random split (same client can appear in both train/test) ---
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    df[feature_cols], df["is_declining"], test_size=0.3, random_state=42
)
rf_random = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42).fit(X_train_r, y_train_r)
scores_random = rf_random.predict_proba(X_test_r)[:, 1]
auc_random = roc_auc_score(y_test_r, scores_random)
p50_random = precision_at_k(scores_random, y_test_r, 50)

# --- AFTER: grouped split by client (honest, from ML-08) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["client_hash_id"]))
train_df2, test_df2 = df.iloc[train_idx], df.iloc[test_idx]

rf_grouped = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42).fit(
    train_df2[feature_cols], train_df2["is_declining"]
)
scores_grouped = rf_grouped.predict_proba(test_df2[feature_cols])[:, 1]
auc_grouped = roc_auc_score(test_df2["is_declining"], scores_grouped)
p50_grouped = precision_at_k(scores_grouped, test_df2["is_declining"], 50)

before_after = pd.DataFrame({
    "split": ["Random (before)", "Grouped by client (after)"],
    "AUC": [round(auc_random, 3), round(auc_grouped, 3)],
    "precision@50": [round(p50_random, 3), round(p50_grouped, 3)]
})
before_after

,split,AUC,precision@50
0,Random (before),0.861,0.84
1,Grouped by client (after),0.804,0.56


**Findings:** The gap between random and grouped splits is large — precision@50
drops from 0.84 to 0.56 (a 33% relative drop), while AUC drops more mildly
(0.861 to 0.804). This confirms the random split was overstating performance:
part of the "before" score came from the model recognizing client-specific
patterns it had already seen in training, not from genuinely predicting
decline on unseen clients. The grouped-split number (precision@50 = 0.56,
AUC = 0.804) is the honest number I should report and trust going forward —
consistent with what I flagged as a methodology question for the paper's own
Growth Prediction model (Finding B), which showed a similar same-brand vs.
new-brand gap.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

##  3: Leakage audit

Re-running the same leakage hunt from ML-05 (`hunting-leakage-and-validating`
checklist) on my final feature set: (1) test with a known label-derived column
added, (2) single-feature AUC scan for suspiciously perfect predictors, (3)
confirm no product/decision flags are used as features.

In [7]:
from sklearn.linear_model import LogisticRegression

# --- Test 1: add a known label-derived column, confirm the score jumps ---
leak_data = con.sql(f"""
    WITH halves AS (
        SELECT client_hash_id, content_hash_id,
            SUM(CASE WHEN report_date < DATE '2026-03-16' THEN gsc_impressions ELSE 0 END) AS first_half,
            SUM(CASE WHEN report_date >= DATE '2026-03-16' THEN gsc_impressions ELSE 0 END) AS second_half
        FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT client_hash_id, content_hash_id, (second_half - first_half) AS impression_change
    FROM halves
""").df()

audit_df = df.merge(leak_data, on=["client_hash_id", "content_hash_id"])

# honest (no leak) on the SAME grouped split as Section 2
X_train_h = train_df2[feature_cols]
X_test_h = test_df2[feature_cols]
y_train_h = train_df2["is_declining"]
y_test_h = test_df2["is_declining"]

honest_auc = roc_auc_score(y_test_h, RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
                            .fit(X_train_h, y_train_h).predict_proba(X_test_h)[:, 1])

# with the leak added
audit_train = audit_df.loc[train_df2.index]
audit_test = audit_df.loc[test_df2.index]
leak_cols = feature_cols + ["impression_change"]

leaked_auc = roc_auc_score(audit_test["is_declining"],
                            RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
                            .fit(audit_train[leak_cols], audit_train["is_declining"])
                            .predict_proba(audit_test[leak_cols])[:, 1])

print("Test 1 — Honest AUC (grouped split):", round(honest_auc, 3))
print("Test 1 — Leaked AUC (with impression_change added):", round(leaked_auc, 3))

# --- Test 2: single-feature AUC scan ---
print("\nTest 2 — Single-feature AUC scan (flag anything > 0.95 or < 0.05):")
for col in feature_cols:
    single_auc = roc_auc_score(df["is_declining"], df[[col]].fillna(0))
    flag = "  <-- SUSPICIOUS" if single_auc > 0.95 or single_auc < 0.05 else ""
    print(f"  {col}: {single_auc:.3f}{flag}")

# --- Test 3: confirm no product/decision flags used ---
print("\nTest 3 — Features used:", feature_cols)
print("Test 3 — Product/decision flags deliberately excluded:",
      ["last_optimized_date", "optimization_eligible_date"])


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Test 1 — Honest AUC (grouped split): 0.804
Test 1 — Leaked AUC (with impression_change added): 1.0

Test 2 — Single-feature AUC scan (flag anything > 0.95 or < 0.05):
  gsc_impressions_total: 0.792
  gsc_clicks_total: 0.607
  gsc_avg_position: 0.789
  ga4_sessions_total: 0.583
  ga4_pageviews_total: 0.584
  ga4_engaged_sessions_total: 0.517
  days_observed: 0.558
  days_gsc_available: 0.814
  ctr: 0.606
  engagement_rate: 0.517
  gsc_coverage: 0.811

Test 3 — Features used: ['gsc_impressions_total', 'gsc_clicks_total', 'gsc_avg_position', 'ga4_sessions_total', 'ga4_pageviews_total', 'ga4_engaged_sessions_total', 'days_observed', 'days_gsc_available', 'ctr', 'engagement_rate', 'gsc_coverage']
Test 3 — Product/decision flags deliberately excluded: ['last_optimized_date', 'optimization_eligible_date']


**Findings:** Test 1 confirms the leakage test harness itself works — adding
`impression_change` pushes AUC to a perfect 1.0, and it remains excluded from
the final feature set. Test 2 shows no individual feature exceeds 0.95 AUC on
its own; `days_gsc_available` (0.814) and `gsc_coverage` (0.811) remain the
highest, consistent with earlier findings — plausible rather than suspicious,
since content with more consistent GSC coverage naturally has more stable
signal. Test 3 confirms no product/decision flags (`last_optimized_date`,
`optimization_eligible_date`) are used as inputs. The final 11-feature set
passes the leakage audit on the honest grouped split.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## Section 4: Claim rewrite

**Original claim (from ML-08, Section 3):** "Both ML models substantially beat
the Week-4 baseline" — based on a precision@50 of 0.66-0.84 without specifying
which split produced it.

**Why this claim goes further than the evidence:** It doesn't distinguish
between the optimistic random-split number and the honest grouped-split
number, and doesn't note that the client-grouped test shows a real, sizeable
drop (0.84 → 0.56 precision@50). Stating "substantially beats baseline"
without that context risks overstating what the model will do on a client it
has never seen.

**Rewritten claim (safe language):** "On a client-grouped, out-of-sample
split, the Random Forest model showed directional improvement over the
Week-4 rule-based baseline (precision@50 = 0.56 vs. the baseline's 0.16),
though this is measured, decision-support evidence on one month of data
(2026-03) — not a guarantee of performance on new clients or future months.
The gap between the random-split number (0.84) and the grouped-split number
(0.56) suggests some of the model's apparent skill on a naive split reflects
client-specific memorization rather than generalizable signal."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.